In [1]:
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 5.4 MB/s eta 0:00:00


In [2]:
import os
import math
import torch
import torch.nn as nn
import numpy as np
from scipy.io import loadmat
from scipy.signal import resample
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.notebook import tqdm

# ==========================================
# 1. FIXED TRANSFORMER ARCHITECTURE
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class AdvancedECGTransformer(nn.Module):
    def __init__(self, in_channels=12, d_model=256, nhead=8, num_classes=5, num_layers=6):
        super().__init__()
        self.d_model = d_model
        
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, d_model // 2, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(d_model // 2),
            nn.GELU(),
            nn.Conv1d(d_model // 2, d_model, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(d_model),
            nn.GELU()
        )
        self.pos_encoder = PositionalEncoding(d_model=d_model)
        
        # Critical deep-transformer fixes: norm_first=True
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, 
            dropout=0.1, activation='gelu', batch_first=True,
            norm_first=True 
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model)
        )
        
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):
        x = self.stem(x) 
        x = x.transpose(1, 2) 
        x = x * math.sqrt(self.d_model) # Prevent features from drowning in positional noise
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x_avg = torch.mean(x, dim=1) # Global Average Pooling routes gradients to all tokens
        return self.head(x_avg)

# ==========================================
# 2. SAFEGUARDED EVALUATION FUNCTION
# ==========================================
def evaluate_model_advanced(model, dataloader, device):
    model.eval()
    all_probs, all_targets = [], []
    
    with torch.no_grad():
        for signals, labels in dataloader:
            signals = signals.to(device, non_blocking=True)
            with autocast('cuda', dtype=torch.bfloat16):
                probs = torch.sigmoid(model(signals)) 
            all_probs.append(probs.cpu().float().numpy())
            all_targets.append(labels.numpy())
            
    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    
    aucs, auprcs = [], []
    superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
    
    # Safely skip classes with zero positive samples in the validation set
    for i in range(len(superclasses)):
        if len(np.unique(all_targets[:, i])) == 2:
            aucs.append(roc_auc_score(all_targets[:, i], all_probs[:, i]))
            auprcs.append(average_precision_score(all_targets[:, i], all_probs[:, i]))
            
    macro_auc = np.mean(aucs) if len(aucs) > 0 else 0.0
    macro_auprc = np.mean(auprcs) if len(auprcs) > 0 else 0.0
        
    return macro_auc, macro_auprc

In [3]:
# ==========================================
# 3. GEORGIA DATABASE (SCIPY LOADER)
# ==========================================
class GeorgiaDataset(Dataset):
    def __init__(self, data_path, target_hz=100):
        super().__init__()
        self.data_path = data_path
        self.superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
        self.target_length = target_hz * 10 
        
        self.snomed_mapping = {
            '426783006': 'NORM', '426177001': 'NORM', '427084000': 'NORM', '427393009': 'NORM',
            '164865005': 'MI', '164861001': 'MI', '54329005': 'MI',
            '164934002': 'STTC', '59931005': 'STTC', '164917005': 'STTC', '111975006': 'STTC',
            '164909002': 'CD', '733534002': 'CD', '59118001': 'CD', '713427006': 'CD', 
            '713426002': 'CD', '445118002': 'CD', '270492004': 'CD', '164947007': 'CD', '6374002': 'CD',
            '164873001': 'HYP', '370355005': 'HYP'
        }

        # Look for the actual .mat data files to ignore hidden macOS '.' files
        self.files = [f[:-4] for f in os.listdir(data_path) if f.endswith('.mat') and not f.startswith('.')]
        
        self.signals = []
        self.labels = []
        self.skipped_files = 0
        
        print(f"Pre-loading Georgia Database using SciPy | Downsampling 500Hz -> {target_hz}Hz...")
        for f in tqdm(self.files, desc="Loading ECGs"):
            mat_path = os.path.join(self.data_path, f + '.mat')
            hea_path = os.path.join(self.data_path, f + '.hea')
            
            try:
                # 1. Read Native MATLAB Signal
                mat_data = loadmat(mat_path)
                
                if 'val' in mat_data:
                    signal = np.array(mat_data['val'], dtype=np.float32)
                else:
                    key = [k for k in mat_data.keys() if not k.startswith('__')][0]
                    signal = np.array(mat_data[key], dtype=np.float32)
                
                if signal.shape[0] == 12:
                    signal = signal.T 
                
                # Standardize voltage and Downsample
                signal = signal / 1000.0
                signal = np.nan_to_num(signal)
                signal = resample(signal, self.target_length, axis=0) 
                
                signal_tensor = torch.tensor(signal, dtype=torch.float32).transpose(0, 1)
                
                # 2. Extract Labels manually ignoring corrupted text formatting
                with open(hea_path, 'r', encoding='utf-8-sig', errors='ignore') as text_file:
                    lines = text_file.readlines()
                
                dx_str = ""
                for line in lines:
                    if line.startswith('#Dx:') or line.startswith('# Dx:'):
                        dx_str = line.split(':')[1].strip()
                        break
                
                if not dx_str:
                    self.skipped_files += 1
                    continue
                
                label = np.zeros(len(self.superclasses))
                for code in dx_str.split(','):
                    code = code.strip()
                    if code in self.snomed_mapping:
                        idx = self.superclasses.index(self.snomed_mapping[code])
                        label[idx] = 1
                        
                self.signals.append(signal_tensor)
                self.labels.append(torch.tensor(label, dtype=torch.float32))
                
            except Exception as e:
                self.skipped_files += 1
                continue

        print(f"Successfully loaded {len(self.signals)} records. Skipped {self.skipped_files} files.")

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

In [4]:
# 1. Setup Directories
GEORGIA_DIR = "/kaggle/input/datasets/organizations/physionet/georgia-12lead-ecg-challenge-database/Georgia"

# 2. Load and Split the Dataset (80% Train, 20% Validation)
full_dataset = GeorgiaDataset(data_path=GEORGIA_DIR, target_hz=100)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

# Locked random seed guarantees the exact same 80/20 split as your Mamba run
generator = torch.Generator().manual_seed(42) 

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

# num_workers=0 remains critical to safeguard Kaggle RAM
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Training on {train_size} records. Validating on {val_size} records.")

# 3. Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the SOTA Transformer baseline
model = AdvancedECGTransformer(
    in_channels=12, d_model=256, nhead=8, num_classes=5, num_layers=6
).to(device)

# Using the lowered 3e-4 Learning Rate for Transformer Stability
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
criterion = nn.BCEWithLogitsLoss()

# 4. The Training Loop
epochs = 30
max_norm = 1.0

print("Starting Transformer Baseline Training on Georgia Database...")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    valid_batches = 0
    
    for i, (signals, labels) in enumerate(train_dataloader):
        signals, labels = signals.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True) 
        
        with autocast('cuda', dtype=torch.bfloat16):
            outputs = model(signals)
            loss = criterion(outputs, labels)
            
        if torch.isnan(loss):
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        optimizer.step()
        total_loss += loss.item()
        valid_batches += 1
        
    scheduler.step()
    avg_train_loss = total_loss / max(valid_batches, 1)
    
    # Live Validation check at the end of each epoch
    val_auc, val_auprc = evaluate_model_advanced(model, val_dataloader, device)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val AUC: {val_auc:.4f} | Val AUPRC: {val_auprc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# Save the weights
torch.save(model.state_dict(), '/kaggle/working/transformer_georgia_weights.pth')
print("Transformer Model saved successfully!")

Pre-loading Georgia Database using SciPy | Downsampling 500Hz -> 100Hz...


Loading ECGs:   0%|          | 0/10344 [00:00<?, ?it/s]

Successfully loaded 10344 records. Skipped 0 files.
Training on 8275 records. Validating on 2069 records.


/tmp/ipykernel_58/279853892.py:51: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Starting Transformer Baseline Training on Georgia Database...
Epoch [1/30] | Train Loss: 0.3968 | Val AUC: 0.8368 | Val AUPRC: 0.6998 | LR: 0.000293
Epoch [2/30] | Train Loss: 0.3295 | Val AUC: 0.8627 | Val AUPRC: 0.7368 | LR: 0.000271
Epoch [3/30] | Train Loss: 0.3092 | Val AUC: 0.8715 | Val AUPRC: 0.7512 | LR: 0.000238
Epoch [4/30] | Train Loss: 0.2940 | Val AUC: 0.8826 | Val AUPRC: 0.7654 | LR: 0.000197
Epoch [5/30] | Train Loss: 0.2746 | Val AUC: 0.8894 | Val AUPRC: 0.7746 | LR: 0.000150
Epoch [6/30] | Train Loss: 0.2587 | Val AUC: 0.8981 | Val AUPRC: 0.7889 | LR: 0.000104
Epoch [7/30] | Train Loss: 0.2372 | Val AUC: 0.9013 | Val AUPRC: 0.7926 | LR: 0.000063
Epoch [8/30] | Train Loss: 0.2200 | Val AUC: 0.9011 | Val AUPRC: 0.7936 | LR: 0.000030
Epoch [9/30] | Train Loss: 0.2050 | Val AUC: 0.9036 | Val AUPRC: 0.7957 | LR: 0.000008
Epoch [10/30] | Train Loss: 0.1947 | Val AUC: 0.9024 | Val AUPRC: 0.7945 | LR: 0.000300
Epoch [11/30] | Train Loss: 0.2446 | Val AUC: 0.9002 | Val AUPRC: 0